In [1]:
from pathlib import Path
from bertopic import BERTopic
import pandas as pd
import plotly.io as pio

df_grid = pd.read_csv("./final/next/next_grid_search_results.csv")
df_grid['avg_score'] = (df_grid['silhouette'] + df_grid['coherence'] + df_grid['diversity']) / 3
df_grid=df_grid[(df_grid['hdbscan_min_cluster_size']!=500) & (df_grid['hdbscan_min_cluster_size']!=30)]
best_models = {
    'silhouette': df_grid.sort_values(by='silhouette', ascending=False).iloc[0],
    'coherence': df_grid.sort_values(by='coherence', ascending=False).iloc[0],
    'diversity': df_grid.sort_values(by='diversity', ascending=False).iloc[0],
    'avg_score': df_grid.sort_values(by='avg_score', ascending=False).iloc[0]
}

topic_models = {}
for key, row in best_models.items():
    suffix = f"{row['model']}_umap{row['umap_n_components']}_umap{row['umap_n_neighbors']}_umap{row['umap_min_dist']}_hdbscan{row['hdbscan_min_cluster_size']}"
    model_path = Path(f"./final/next/next_bertopic_models/{suffix}")
    
    if not model_path.exists():
        print(f"Path not found: {model_path}")
        continue
    
    print(f"Loading model for best {key} from: {model_path}")

    topic_model = BERTopic.load(model_path)
    topic_models[key] = topic_model
    num_topics=len(set(topic_model.topics_))
    print(f"{num_topics} topics found")
    # Genera grafico
    fig = topic_model.visualize_barchart(top_n_topics=min(80,num_topics), n_words=20, width=350, height=450)
    fig.show()
    # Salva grafico come immagine
    #out_path = Path(f"../final/barchart_{key}.png")
    #pio.write_image(fig, out_path)
    #print(f"Saved chart to {out_path}")


/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model for best silhouette from: final/next/next_bertopic_models/paraphrase-MiniLM-L6-v2_umap5_umap25_umap0.0_hdbscan250


ModuleNotFoundError: No module named 'cuml'

In [ ]:
df=pd.read_csv('./final/next/next_df_sampled.csv')
df

In [ ]:
texts = df['text_preprocessed']
timestamps = pd.to_datetime(pd.to_datetime(df['date'],format="%Y-%m-%d %H:%M:%S").dt.date)

topics_over_time = topic_models['silhouette'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)


In [ ]:
topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)

In [ ]:
topic_models['silhouette'].get_topic_info()

In [ ]:

topic_models['silhouette'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

In [ ]:
df['topic']=topic_models['silhouette'].topics_

In [ ]:
df[df['topic']==1]